In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.metropulse;
CREATE VOLUME IF NOT EXISTS workspace.metropulse.raw;

In [0]:
AWS_KEY_ID = dbutils.secrets.get(scope="metropulse", key="aws-access-key-id")
AWS_SECRET = dbutils.secrets.get(scope="metropulse", key="aws-secret-access-key")

BUCKET = "metropulse-raw-muhammad"
PREFIX = "raw/gtfs/"
REGION = "us-east-1"
VOLUME_PATH = "/Volumes/workspace/metropulse/raw/gtfs"

print(len(AWS_KEY_ID), len(AWS_SECRET))

20 40


In [0]:
import boto3

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY_ID,
    aws_secret_access_key=AWS_SECRET,
    region_name=REGION,
)

response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)
files = [obj for obj in response.get("Contents", []) if not obj["Key"].endswith("/")]

for obj in files:
    print(f"{obj['Key']:45} {obj['Size']:>12,} bytes")
print(f"\n{len(files)} files found")

raw/gtfs/agency.txt                                    243 bytes
raw/gtfs/calendar.txt                                  305 bytes
raw/gtfs/calendar_dates.txt                            760 bytes
raw/gtfs/direction_names_exceptions.txt              7,427 bytes
raw/gtfs/fare_attributes.txt                           261 bytes
raw/gtfs/fare_rules.txt                              1,761 bytes
raw/gtfs/feed_info.txt                                 213 bytes
raw/gtfs/routes.txt                                 20,199 bytes
raw/gtfs/shapes.txt                              9,297,934 bytes
raw/gtfs/stop_times.txt                         71,516,497 bytes
raw/gtfs/stops.txt                                 812,500 bytes
raw/gtfs/trips.txt                               1,365,749 bytes

12 files found


In [0]:
import os

os.makedirs(VOLUME_PATH, exist_ok=True)

for obj in files:
    key = obj["Key"]
    filename = key.split("/")[-1]
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"]

    with open(f"{VOLUME_PATH}/{filename}", "wb") as f:
        for chunk in body.iter_chunks(chunk_size=8 * 1024 * 1024):
            f.write(chunk)

    print(f"Copied {filename}")

Copied agency.txt
Copied calendar.txt
Copied calendar_dates.txt
Copied direction_names_exceptions.txt
Copied fare_attributes.txt
Copied fare_rules.txt
Copied feed_info.txt
Copied routes.txt
Copied shapes.txt
Copied stop_times.txt
Copied stops.txt
Copied trips.txt


In [0]:
display(dbutils.fs.ls(VOLUME_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/metropulse/raw/gtfs/agency.txt,agency.txt,243,1793646585000
dbfs:/Volumes/workspace/metropulse/raw/gtfs/calendar.txt,calendar.txt,305,1793646586000
dbfs:/Volumes/workspace/metropulse/raw/gtfs/calendar_dates.txt,calendar_dates.txt,760,1793646586000
dbfs:/Volumes/workspace/metropulse/raw/gtfs/direction_names_exceptions.txt,direction_names_exceptions.txt,7427,1793646587000
dbfs:/Volumes/workspace/metropulse/raw/gtfs/fare_attributes.txt,fare_attributes.txt,261,1793646587000
dbfs:/Volumes/workspace/metropulse/raw/gtfs/fare_rules.txt,fare_rules.txt,1761,1793646588000
dbfs:/Volumes/workspace/metropulse/raw/gtfs/feed_info.txt,feed_info.txt,213,1793646588000
dbfs:/Volumes/workspace/metropulse/raw/gtfs/routes.txt,routes.txt,20199,1793646588000
dbfs:/Volumes/workspace/metropulse/raw/gtfs/shapes.txt,shapes.txt,9297934,1793646590000
dbfs:/Volumes/workspace/metropulse/raw/gtfs/stop_times.txt,stop_times.txt,71516497,1793646590000
